# E-Commerce Hybrid Recommendation System
**Dataset**: Olist Brazilian E-Commerce (Kaggle, CC BY-NC-SA 4.0)

## Phase 1 — Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
DATA_PATH = "data/raw/"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

## Phase 2 — Data Loading & Cleaning

In [2]:
# Load all 8 CSVs
df_orders = pd.read_csv(
    DATA_PATH + "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

df_items = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
df_products = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
df_customers = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
df_reviews = pd.read_csv(
    DATA_PATH + "olist_order_reviews_dataset.csv",
    parse_dates=["review_creation_date", "review_answer_timestamp"],
)
df_payments = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
df_sellers = pd.read_csv(DATA_PATH + "olist_sellers_dataset.csv")
df_geolocation = pd.read_csv(DATA_PATH + "olist_geolocation_dataset.csv")

print("Loaded:")
for name, df in [
    ("df_orders", df_orders), ("df_items", df_items), ("df_products", df_products),
    ("df_customers", df_customers), ("df_reviews", df_reviews), ("df_payments", df_payments),
    ("df_sellers", df_sellers), ("df_geolocation", df_geolocation),
]:
    print(f"  {name:20s} {df.shape}")

Loaded:
  df_orders            (99441, 8)
  df_items             (112650, 7)
  df_products          (32951, 9)
  df_customers         (99441, 5)
  df_reviews           (99224, 7)
  df_payments          (103886, 5)
  df_sellers           (3095, 4)
  df_geolocation       (1000163, 5)


In [3]:
# Filter to delivered orders only
df_orders = df_orders[df_orders["order_status"] == "delivered"].reset_index(drop=True)
print(f"df_orders after delivered filter: {df_orders.shape}  (expected ~96478)")

df_orders after delivered filter: (96478, 8)  (expected ~96478)


In [4]:
# Null handling

# products: drop 610 rows with null product_category_name (can't use in CF matrix or CB embeddings)
df_products = df_products.dropna(subset=["product_category_name"]).reset_index(drop=True)

# products: fill product_description_lenght nulls with 0
df_products["product_description_lenght"] = (
    df_products["product_description_lenght"].fillna(0).astype(int)
)

# payments: multiple rows per order (installments, vouchers, etc.) — collapse to one row per order
df_payments_agg = (
    df_payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
    )
)

print(f"df_products after null drop: {df_products.shape}  (expected ~32341)")
print(f"df_payments_agg:             {df_payments_agg.shape}  (one row per order)")

df_products after null drop: (32341, 9)  (expected ~32341)
df_payments_agg:             (99440, 3)  (one row per order)


In [ ]:
# Add price_bucket and weight_bucket to df_products (needed for CB metadata strings in Phase 6)
price_median = df_items.groupby("product_id")["price"].median().rename("price_median")
df_products = df_products.join(price_median, on="product_id")

# Use "unknown" for products with no price history (no items ever listed)
# fillna(0) would incorrectly label them "budget" in CB embeddings
df_products["price_bucket"] = pd.cut(
    df_products["price_median"],
    bins=[0, 50, 150, 500, float("inf")],
    labels=["budget", "mid", "premium", "luxury"],
    include_lowest=True,
)
df_products["price_bucket"] = df_products["price_bucket"].cat.add_categories("unknown")
df_products["price_bucket"] = df_products["price_bucket"].fillna("unknown")

df_products["weight_bucket"] = pd.cut(
    df_products["product_weight_g"],
    bins=[0, 500, 2000, 10000, float("inf")],
    labels=["light", "medium", "heavy", "bulky"],
    include_lowest=True,
)
df_products["weight_bucket"] = df_products["weight_bucket"].cat.add_categories("unknown")
df_products["weight_bucket"] = df_products["weight_bucket"].fillna("unknown")

print(df_products[["product_id", "product_category_name", "price_bucket", "weight_bucket"]].head(8))

In [ ]:
# Build df_master: orders → items → products → customers → reviews → payments
# NOTE: df_master is ORDER-ITEM level — one row per (order × item).
# For order-level aggregations in EDA (monthly counts, payment stats), use:
#   df_orders_view = df_master.drop_duplicates("order_id")
df_master = (
    df_orders
    .merge(df_items, on="order_id", how="inner")
    .merge(df_products, on="product_id", how="inner")
    .merge(df_customers, on="customer_id", how="inner")
    .merge(
        # Sort by most recent answer to keep the latest review per order (deterministic)
        df_reviews[["order_id", "review_score", "review_answer_timestamp"]]
            .sort_values("review_answer_timestamp", ascending=False, na_position="last")
            .drop_duplicates("order_id")[["order_id", "review_score"]],
        on="order_id",
        how="left",
    )
    .merge(df_payments_agg, on="order_id", how="left")
)

print(f"df_master shape: {df_master.shape}")
df_master.head(3)

In [ ]:
# Verify — primary-key uniqueness
assert df_orders["order_id"].is_unique, "Duplicate order_ids after delivered filter"
assert df_customers["customer_id"].is_unique, "Duplicate customer_ids in df_customers"
assert df_orders.shape[0] > 90_000, f"Expected >90k delivered orders, got {df_orders.shape[0]}"
assert pd.api.types.is_datetime64_any_dtype(df_orders["order_purchase_timestamp"]), \
    "Timestamp not parsed as datetime"

# These columns must NEVER be null in df_master (they come from inner-joined tables)
required_non_null = ["order_id", "customer_id", "product_id", "product_category_name", "price"]
for col in required_non_null:
    null_count = df_master[col].isnull().sum()
    assert null_count == 0, f"Expected 0 nulls in df_master[\'{col}\'], got {null_count}"

print("All assertions passed.")

print("\n=== Shape Summary ===")
for name, df in [
    ("df_orders", df_orders), ("df_items", df_items), ("df_products", df_products),
    ("df_customers", df_customers), ("df_master", df_master),
]:
    print(f"  {name:20s} {df.shape}")

print("\n=== df_master null counts (non-zero only) ===")
nulls = df_master.isnull().sum()
nulls_nonzero = nulls[nulls > 0]
print(nulls_nonzero if len(nulls_nonzero) else "  none")
print("  (review_score and payment_value nulls are expected: left joins)")

print("\n=== df_master dtypes ===")
print(df_master.dtypes)

### Cleaning Decisions

**Filter (`order_status == "delivered"`)**: Removes 2,963 non-delivered orders (cancelled, invoiced, in-transit, etc.). Only delivered orders carry meaningful review scores and actual purchase signals.

**Product null drops (610 rows)**: Rows with no `product_category_name` are unusable for both the CF user-item matrix (which is keyed on category) and CB embeddings (which need a category string). Dropping them is safer than imputing an unknown category.

**`product_description_lenght` fill 0**: After dropping null-category rows no nulls remain here, but filling with 0 is defensive — a missing description length is effectively zero content.

**Payments aggregation**: The payments table stores one row per payment method (credit card + voucher = 2 rows). We sum `payment_value` and take the max `payment_installments` to get one row per order, preventing row fan-out during the join.

**Reviews dedup**: A small number of orders have duplicate review rows. We sort by `review_answer_timestamp` descending and keep the most recent review per order — deterministic and semantically correct (latest opinion wins).

**Sellers & Geolocation excluded from df_master**: `df_geolocation` has 1M rows with duplicate zip codes — joining it directly would fan-out df_master. Both tables are available for targeted lookups in later phases (e.g., RQ1 uses `customer_state` from `df_customers` directly).

**df_master row count**: Higher than `df_orders` because an order can contain multiple items (`df_items` is item-level, not order-level).

## Phase 3 — Exploratory Data Analysis

In [ ]:
# --- 3.1 Monthly order volume ---
df_monthly = (
    df_master.drop_duplicates("order_id")
    .set_index("order_purchase_timestamp")
    .resample("ME")["order_id"]
    .count()
    .reset_index()
    .rename(columns={"order_id": "order_count", "order_purchase_timestamp": "month"})
)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_monthly["month"], df_monthly["order_count"], marker="o", linewidth=1.5)
ax.set_title("Monthly Order Volume (Delivered Orders)")
ax.set_xlabel("Month")
ax.set_ylabel("Order Count")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

Order volume grew steadily from late 2016 through late 2017, peaked in November 2017 (Black Friday effect), and remained near-peak through mid-2018. The first two months (Sep–Oct 2016) and the final month (Aug 2018) show low counts due to partial coverage — these are dataset boundary artifacts, not real drops. The dataset covers approximately 23 months of transactions.

In [ ]:
# --- 3.2 Top-15 product categories (English labels) ---
df_cat_trans = pd.read_csv(DATA_PATH + "product_category_name_translation.csv")
cat_name_map = df_cat_trans.set_index("product_category_name")["product_category_name_english"].to_dict()

top_cats = (
    df_master.drop_duplicates(["order_id", "product_category_name"])["product_category_name"]
    .value_counts()
    .head(15)
    .rename(index=cat_name_map)
    .sort_values()
)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_cats.index, top_cats.values, color="steelblue")
ax.set_title("Top 15 Product Categories (orders containing category)")
ax.set_xlabel("Number of Orders")
plt.tight_layout()
plt.show()

Bed/bath/table linen and health/beauty lead by a wide margin, together accounting for ~18% of all category-order pairs. Computers & accessories and sports/leisure also feature prominently. The distribution is heavily right-skewed — the top 5 categories capture nearly half of all orders. This concentration implies cold-start users can be given reasonable content-based recommendations even without prior purchase history.

In [ ]:
# --- 3.3 Review score distribution ---
review_scores = df_master.drop_duplicates("order_id")["review_score"].dropna()
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(review_scores, bins=5, kde=True, ax=ax, discrete=True)
ax.set_title("Review Score Distribution")
ax.set_xlabel("Review Score (1–5)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

Reviews are strongly skewed toward 5 stars (~60% of all reviews). Scores 1 and 2 together account for less than 15%. This positive skew is typical for e-commerce platforms. For the CF model, this means the global-mean baseline will predict ~4.2 for most interactions — the SVD must meaningfully outperform this to justify the added complexity.

In [ ]:
# --- 3.4 Payment value by review score ---
df_pay = df_master.drop_duplicates("order_id")[["payment_value", "review_score"]].dropna()
df_pay["review_score"] = df_pay["review_score"].astype(int)
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df_pay, x="review_score", y="payment_value", ax=ax,
            flierprops={"marker": ".", "alpha": 0.3})
ax.set_ylim(0, 600)  # cap outliers for readability
ax.set_title("Payment Value by Review Score (outliers >R$600 not shown)")
ax.set_xlabel("Review Score")
ax.set_ylabel("Payment Value (R$)")
plt.tight_layout()
plt.show()

1-star orders have the highest median payment value (~R$121) while scores 2–5 cluster around R$103–112. This suggests high-value orders are more likely to disappoint — possibly because expensive purchases come with higher expectations or longer delivery times. Outliers above R$600 are not shown; the pattern holds within the visible range.

In [ ]:
# --- 3.5 Unique customer count by Brazilian state ---
# customer_unique_id is Olist's stable customer identity (customer_id is order-scoped)
state_counts = (
    df_master.drop_duplicates("customer_unique_id")["customer_state"]
    .value_counts()
)
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(state_counts.index, state_counts.values, color="teal")
ax.set_title("Unique Customer Count by Brazilian State")
ax.set_xlabel("State")
ax.set_ylabel("Unique Customers")
plt.tight_layout()
plt.show()

São Paulo (SP) accounts for ~42% of unique customers, followed by Rio de Janeiro (RJ, ~13%) and Minas Gerais (MG, ~12%). The Southeast region dominates, reflecting Brazil's population distribution. For RQ1, states outside the top 5 have small sample sizes — we will focus the violin plot on the top-5 states to maintain statistical validity.

## Phase 4 — RQ1: Regional Rating Bias by Customer Type

**Question**: Do multi-category buyers rate products differently by Brazilian region?

In [ ]:
# Count distinct categories purchased per stable customer identity
cat_counts = (
    df_master
    .dropna(subset=['product_category_name'])
    .groupby('customer_unique_id')['product_category_name']
    .nunique()
    .reset_index()
    .rename(columns={'product_category_name': 'n_categories'})
)
cat_counts['is_multi_category'] = cat_counts['n_categories'] >= 2

# Deduplicate to order level once — reused in next cell for top5_states
df_orders_unique = df_master.drop_duplicates('order_id')

# One row per order (review is order-scoped), drop missing review_score
df_rq1 = (
    df_orders_unique[
        ['order_id', 'customer_unique_id', 'customer_state', 'review_score']
    ]
    .dropna(subset=['review_score'])
    .merge(cat_counts[['customer_unique_id', 'is_multi_category']], on='customer_unique_id')
)
df_rq1['buyer_type'] = df_rq1['is_multi_category'].map(
    {True: 'Multi-Category', False: 'Single-Category'}
)

print(f"df_rq1 shape: {df_rq1.shape}")
print(df_rq1['buyer_type'].value_counts())

In [ ]:
mean_by_type_state = (
    df_rq1.groupby(['customer_state', 'buyer_type'])['review_score']
    .mean()
    .unstack()
    .round(3)
)
# Top-5 states by order count (reuses df_orders_unique from previous cell)
top5_states = (
    df_orders_unique['customer_state']
    .value_counts().head(5).index.tolist()
)
print("Mean review score (top-5 states):")
print(mean_by_type_state.loc[top5_states])

In [ ]:
df_plot = df_rq1[df_rq1['customer_state'].isin(top5_states)].copy()

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(
    data=df_plot,
    x='customer_state',
    y='review_score',
    hue='buyer_type',
    split=True,
    inner='quart',
    cut=0,          # clip KDE at data range [1,5] — scores are discrete ordinal
    palette={'Multi-Category': '#2196F3', 'Single-Category': '#FF9800'},
    ax=ax,
    order=top5_states,
)
ax.set_title('RQ1: Review Score Distribution — Top-5 States by Buyer Type')
ax.set_xlabel('Customer State')
ax.set_ylabel('Review Score')
ax.legend(title='Buyer Type')
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

# Aggregate to customer level (one mean score per customer)
# Multi-cat customers have 2+ orders by definition → order-level violates independence
cust_scores = (
    df_rq1.groupby(['customer_unique_id', 'is_multi_category'])['review_score']
    .mean()
    .reset_index()
)
multi_scores  = cust_scores[cust_scores['is_multi_category']]['review_score']
single_scores = cust_scores[~cust_scores['is_multi_category']]['review_score']

stat, p_value = stats.mannwhitneyu(multi_scores, single_scores, alternative='two-sided')

# Effect size r = Z / sqrt(N) — computed from U directly to avoid p-value underflow at large N
n1, n2 = len(multi_scores), len(single_scores)
n_total = n1 + n2
mu_u = n1 * n2 / 2
sigma_u = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
z_score = (stat - mu_u) / sigma_u
effect_r = abs(z_score) / np.sqrt(n_total)

print(f"Mann-Whitney U Test — multi-category vs single-category buyers (customer-level)")
print(f"  U statistic : {stat:.0f}")
print(f"  p-value     : {p_value:.2e}")
print(f"  n_multi     : {n1:,} customers")
print(f"  n_single    : {n2:,} customers")
print(f"  Z score     : {z_score:.4f}")
print(f"  Effect size r: {effect_r:.4f}  (< 0.1 negligible, 0.1–0.3 small, > 0.3 medium)")
sig = p_value < 0.05
print(f"  Result      : {'Statistically significant (α=0.05)' if sig else 'Not significant (α=0.05)'}")

Multi-category buyers rate slightly lower than single-category buyers across all top-5 Brazilian states (SP: 4.20 vs 4.25; RJ: 3.82 vs 3.97; MG: 4.08 vs 4.19; RS: 3.91 vs 4.20; PR: 4.09 vs 4.25). The Mann-Whitney U test (customer-level, n=2,154 multi vs n=89,347 single) is statistically significant (p ≈ 1.4×10⁻³³), but the effect size is negligible (r ≈ 0.04) — with ~90k single-category customers dwarfing ~2k multi-category customers, a significant p-value is near-guaranteed even for trivially small differences; effect size is the meaningful metric here. **The practical conclusion is: multi-category buyers rate slightly but not meaningfully lower.** Note: the test is global across all customers; per-state trends are visible in the violin plot above but not formally tested. The `is_multi_category` flag is a lifetime segment — a customer's review may precede their second-category purchase, introducing minor temporal leakage. Confounders (delivery time, price, freight, seller geography) are not controlled.